# How do you score a match between two lists of numbers?


Your music app just played you a song you love, by a band you've never heard of. No human picked it. Somewhere in a data center, two lists of numbers got compared, and the comparison said: same taste.

One list stood for you. One list stood for the song. Something turned the pair into a single number, and that number was high enough to hit play.

Hold that question before we reach for any definition: how do you score a match between two lists of numbers? We're going to build the answer with code, not memorize it.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

you = np.array([4, 1])    # how much you lean toward each quality
song = np.array([3, 2])   # how much the song leans the same way

print("you  :", you)
print("song :", song)

A taste is an arrow. Each entry is a quality, like "energetic" or "mellow", and the number along it is how hard something leans that way.

Below is one probe arrow (that's you) and four candidate arrows (four songs). The rule for scoring a match: multiply the two arrows entry by entry, then add up the products. One multiply, one add, one score.

Before you run the next cell: call it. Which candidate ends up with the highest score against the probe? Does any score go negative?

In [ ]:
candidates = {
    "song_bright": np.array([4, 3]),
    "song_mellow": np.array([-3, 4]),
    "song_moody": np.array([-4, -3]),
    "song_offbeat": np.array([1, -1]),
}
probe = np.array([4, 3])

print("probe:", probe)
for name, vec in candidates.items():
    print(f"{name:14s}: {vec}")

In [ ]:
def dot_score(a, b):
    """Multiply entry by entry, then add. That's the whole rule."""
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

scores = {name: dot_score(probe, vec) for name, vec in candidates.items()}

for name, score in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:14s}: {score:+.1f}")

# cross-check the hand-rolled version against numpy's own dot
for name, vec in candidates.items():
    assert dot_score(probe, vec) == np.dot(probe, vec)

winner = max(scores, key=scores.get)
assert winner == "song_bright"
assert any(s < 0 for s in scores.values())

song_bright wins, because it points the same way as the probe. song_moody goes negative.

Now turn the probe and watch the scores answer. Point at a candidate and its score climbs. Turn away and it dies. So a big score means two songs are the *same*, right?

Before you run the next cell: song_bright never moves below. Only the probe turns. Predict whether its score can still swing from high, to zero, to negative.

In [ ]:
target = candidates["song_bright"]  # this candidate never changes below

for theta in np.linspace(0, 2 * np.pi, 9):
    turning_probe = 5 * np.array([np.cos(theta), np.sin(theta)])
    score = dot_score(turning_probe, target)
    print(f"theta={theta:4.2f} rad   probe={np.round(turning_probe, 2)}   score={score:+.2f}")

The candidate never moved. Only the probe's direction changed, and the score swung from strongly positive to zero to negative anyway. The score isn't reading whether two songs are the same. It's reading whether they agree in direction.

Point the probe along a candidate and the sum climbs to its biggest. Point it across, at a right angle, and the sum dies to exactly zero. Point it against and the sum goes negative. One multiply-and-add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

Two lists of numbers go in. One honest reading of alignment comes out. It has a symbol now: $a \cdot b$, multiply-pairwise-and-add.

In [ ]:
target_angle = np.arctan2(target[1], target[0])

probe_along = 5 * np.array([np.cos(target_angle), np.sin(target_angle)])
probe_across = 5 * np.array([np.cos(target_angle + np.pi / 2), np.sin(target_angle + np.pi / 2)])
probe_against = 5 * np.array([np.cos(target_angle + np.pi), np.sin(target_angle + np.pi)])

agree = dot_score(probe_along, target)
ignore = dot_score(probe_across, target)
oppose = dot_score(probe_against, target)

print(f"agree  (along)  : {agree:+.2f}")
print(f"ignore (across) : {ignore:+.2f}")
print(f"oppose (against): {oppose:+.2f}")

# the meter reading: positive means agree, ~zero means ignore, negative means oppose
assert agree > 0
assert np.isclose(ignore, 0, atol=1e-9)
assert oppose < 0
assert np.isclose(agree, -oppose)  # opposite direction, mirror reading

Rebuild the meter from memory: two arrows go in, so what two operations turn them into the score?

While that settles, one detour worth naming. Swap the songs for a job posting. Score yourself against what the role wants: Python hours, SQL reps, nights free. Multiply the matched entries, add them up, one number for the fit. Different aisle, same meter.

In [ ]:
you_profile = np.array([30, 10, 4])     # python hours, sql reps, nights free
role_wants = np.array([25, 15, 5])      # what the posting asks for
other_candidate = np.array([2, 40, 0])  # heavy sql, no python, no free nights

your_fit = dot_score(you_profile, role_wants)
other_fit = dot_score(other_candidate, role_wants)

print(f"your fit  : {your_fit:.1f}")
print(f"other fit : {other_fit:.1f}")

assert your_fit > other_fit

Same meter, same math, no idea what "python hours" or "nights free" even mean. That's exactly why it travels.

Now the honest look. Take one candidate and stretch it longer, without turning it at all. Its direction holds dead still.

Before you run the next cell: predict what happens to its score against the probe. Does stretching change the direction? Does it change the score?

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)

stretched_song = candidates["song_bright"] * 3  # same direction, three times the length

assert np.allclose(unit(stretched_song), unit(candidates["song_bright"]))

original_score = dot_score(probe, candidates["song_bright"])
stretched_score = dot_score(probe, stretched_song)

print(f"original direction : {unit(candidates['song_bright']).round(3)}")
print(f"stretched direction: {unit(stretched_song).round(3)}")
print(f"original score     : {original_score:+.2f}")
print(f"stretched score    : {stretched_score:+.2f}")

assert stretched_score > original_score

A long, loud song wins on sheer size, not on agreement. A padded resume wins on sheer length. The meter can't yet tell "points the same way" apart from "just bigger."

For the walk home: what is the cheapest fix that keeps the direction and forgets the size?

One thing to try next: write a function that divides a vector by its own length before scoring it, so stretching a candidate no longer changes its reading. Confirm with an assert that the new score is the same for `song_bright` and its stretched copy.